# Note
This notebook is not updated. Refer to descriptives_no_interview.ipynb for most recent analysis.

# Motivating Question


What are the more relevant contexts around the top terms?

In [18]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

# Load the option 1 and option 2 documents
option1_docs = pd.read_csv('../outputs/analysis_results/no_interview/option1/option1_no_interviews.csv')
option2_docs = pd.read_csv('../outputs/analysis_results/no_interview/option2/option2_no_interviews.csv')

print(f"Loaded {len(option1_docs)} Option 1 documents")
print(f"Loaded {len(option2_docs)} Option 2 documents")


Loaded 2017 Option 1 documents
Loaded 2653 Option 2 documents


In [19]:
def find_top_documents_for_term(documents, term, top_n=5, text_column='cleaned_content'):
    """
    Find top N documents most relevant to a specific term using TF-IDF similarity
    
    Parameters:
    - documents: DataFrame containing the documents
    - term: string, the term to search for
    - top_n: int, number of top documents to return
    - text_column: string, name of the column containing document text (default: 'cleaned_content')
    
    Returns:
    - DataFrame with top documents and their relevance scores (includes both original and cleaned content)
    """
    
    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(
        max_features=10000,
        stop_words='english',
        lowercase=True,
        ngram_range=(1, 2)  # Include both unigrams and bigrams
    )
    
    # Fit TF-IDF on all documents
    tfidf_matrix = vectorizer.fit_transform(documents[text_column].astype(str))
    
    # Create a query vector for the term
    term_vector = vectorizer.transform([term])
    
    # Calculate cosine similarity between term and all documents
    similarities = cosine_similarity(term_vector, tfidf_matrix).flatten()
    
    # Get top N document indices
    top_indices = similarities.argsort()[-top_n:][::-1]
    
    # Create result DataFrame
    results = []
    for idx in top_indices:
        doc_text_cleaned = documents.iloc[idx][text_column]
        doc_text_original = documents.iloc[idx]['content'] if 'content' in documents.columns else doc_text_cleaned
        similarity_score = similarities[idx]
        
        # Count term occurrences (case-insensitive) in cleaned text
        term_count = len(re.findall(r'\b' + re.escape(term.lower()) + r'\b', 
                                   doc_text_cleaned.lower()))
        
        results.append({
            'document_index': idx,
            'similarity_score': similarity_score,
            'term_count': term_count,
            'filename': documents.iloc[idx].get('filename', f'doc_{idx}'),
            'cleaned_preview': doc_text_cleaned[:200] + '...' if len(doc_text_cleaned) > 200 else doc_text_cleaned,
            'original_preview': doc_text_original[:200] + '...' if len(doc_text_original) > 200 else doc_text_original,
            'full_cleaned_document': doc_text_cleaned,
            'full_original_document': doc_text_original
        })
    
    return pd.DataFrame(results)


In [20]:
# Function to analyze multiple key terms at once
def analyze_multiple_terms(documents, terms_list, top_n=5):
    """
    Analyze multiple terms and find top documents for each
    
    Parameters:
    - documents: DataFrame containing the documents
    - terms_list: list of terms to analyze
    - top_n: number of top documents per term
    
    Returns:
    - Dictionary with results for each term
    """
    results = {}
    
    for term in terms_list:
        print(f"\n{'='*60}")
        print(f"ANALYZING TERM: '{term.upper()}'")
        print(f"{'='*60}")
        
        term_results = find_top_documents_for_term(documents, term, top_n)
        results[term] = term_results
        
        for i, row in term_results.iterrows():
            print(f"\n--- Document {i+1} (Index: {row['document_index']}) ---")
            print(f"Filename: {row['filename']}")
            print(f"Similarity Score: {row['similarity_score']:.4f}")
            print(f"Term Count: {row['term_count']}")
            print(f"Cleaned Preview: {row['cleaned_preview']}")
            print(f"Original Preview: {row['original_preview']}")
            
    return results


In [21]:
# Analyze key terms for Option 1 and Option 2
key_terms_option1 = ['catholic', 'de valera', 'éamon de valera', 'sinn féin']
key_terms_option2 = ['violence', 'support', 'catholic', 'civil rights', 'sinn féin','brian faulkner','ian paisley','hunger strike']

print("Analyzing multiple key terms from Option 1 documents...")
all_results_option1 = analyze_multiple_terms(option1_docs, key_terms_option1, top_n=5)

print("Analyzing multiple key terms from Option 2 documents...")
all_results_option2 = analyze_multiple_terms(option2_docs, key_terms_option2, top_n=5)

Analyzing multiple key terms from Option 1 documents...

ANALYZING TERM: 'CATHOLIC'

--- Document 1 (Index: 404) ---
Filename: Updated Doherty (2001) no textboxes.docx
Similarity Score: 0.4765
Term Count: 7
Cleaned Preview: idered catholic faith satanic consistently described successive pope roman catholic generally looked upon loyal primarily pope rome rather united kingdom meant depicted potentially untrustworthy trait...
Original Preview: idered the Catholic faith satanic, and consistently described successive popes as the 'anti-Christ'. Roman Catholics were generally looked upon as being loyal primarily to the pope in Rome rather than...

--- Document 2 (Index: 1110) ---
Filename: option1_combined all.docx
Similarity Score: 0.3789
Term Count: 7
Cleaned Preview: substance fear put rule unionist majority separate state explain catholic fearful future event summer august violence continued spiral august serious consequence catholic people lisburn chapel hill li...
Original Preview: su

In [ ]:
# Save results to CSV 
def save_results_to_csv(results_dict, output_dir='../outputs/analysis_results/'):
    """
    Save the term analysis results to CSV files
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    for term, df in results_dict.items():
        filename = f"{output_dir}top_documents_{term.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False)
        print(f"Saved results for '{term}' to {filename}")

# Save results for both options
print("Saving Option 1 results...")
save_results_to_csv(all_results_option1, '../outputs/analysis_results/no_interview/sentiments/option1/')

print("Saving Option 2 results...")
save_results_to_csv(all_results_option2, '../outputs/analysis_results/no_interview/sentiments/option2/')


Saving Option 1 results...
Saved results for 'catholic' to ../outputs/analysis_results/no_interview/sentiments/option1/top_documents_catholic.csv
Saved results for 'de valera' to ../outputs/analysis_results/no_interview/sentiments/option1/top_documents_de_valera.csv
Saved results for 'éamon de valera' to ../outputs/analysis_results/no_interview/sentiments/option1/top_documents_éamon_de_valera.csv
Saved results for 'sinn féin' to ../outputs/analysis_results/no_interview/sentiments/option1/top_documents_sinn_féin.csv
Saving Option 2 results...
Saved results for 'violence' to ../outputs/analysis_results/no_interview/sentiments/option2/top_documents_violence.csv
Saved results for 'support' to ../outputs/analysis_results/no_interview/sentiments/option2/top_documents_support.csv
Saved results for 'catholic' to ../outputs/analysis_results/no_interview/sentiments/option2/top_documents_catholic.csv
Saved results for 'civil rights' to ../outputs/analysis_results/no_interview/sentiments/option2/t

sentiment of the key content?

In [24]:
# Install required packages (run this once)
# !pip install langchain langchain-openai python-dotenv

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema import HumanMessage
import time
import json

# Load environment variables (create a .env file with your OPENAI_API_KEY)
load_dotenv()

# Initialize OpenAI client
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,  # Low temperature for consistent sentiment analysis
    max_tokens=100
)


In [ ]:
# Debugged version
def analyze_sentiment_with_openai_debug(text, search_term, max_length=1000):
    """
    Analyze sentiment of text content using OpenAI via LangChain
    
    Parameters:
    - text: str, the text to analyze
    - search_term: str, the term being searched for context
    - max_length: int, maximum text length to send to OpenAI (to control costs)
    
    Returns:
    - dict with sentiment score, label, and reasoning
    
    Debug version with better error handling and response parsing
    """
    
    # Truncate text if too long
    if len(text) > max_length:
        text = text[:max_length] + "..."
    
    # prompt
    prompt = f"""
Analyze the sentiment of this text about "{search_term}":

"{text}"

Rate the sentiment from -1.0 (very negative) to 1.0 (very positive).
Also provide a label: Very Negative, Negative, Neutral, Positive, or Very Positive.

Response format:
Score: [number]
Label: [label]
Reason: [brief explanation]
"""
    
    try:
        # Get response from OpenAI
        response = llm.invoke([HumanMessage(content=prompt)])
        response_text = response.content.strip()
        
        print(f"Raw OpenAI response: {response_text[:200]}...")  # Debug output
        
        # Parse the response manually instead of expecting JSON
        lines = response_text.split('\n')
        score = 0.0
        label = "Neutral"
        reason = "No reasoning provided"
        
        for line in lines:
            line = line.strip()
            if line.startswith('Score:'):
                try:
                    score_str = line.replace('Score:', '').strip()
                    score = float(score_str)
                except:
                    score = 0.0
            elif line.startswith('Label:'):
                label = line.replace('Label:', '').strip()
            elif line.startswith('Reason:'):
                reason = line.replace('Reason:', '').strip()
        
        return {
            'sentiment_score': score,
            'sentiment_label': label,
            'reasoning': reason,
            'api_success': True
        }
        
    except Exception as e:
        print(f"Error in sentiment analysis: {str(e)}")
        print(f"Response content: {response.content if 'response' in locals() else 'No response'}")
        return {
            'sentiment_score': 0.0,
            'sentiment_label': 'Error',
            'reasoning': f'API Error: {str(e)}',
            'api_success': False
        }


Testing debug sentiment analysis...
Raw OpenAI response: Score: -0.8  
Label: Very Negative  
Reason: The text describes a situation where the Catholic community experienced "discrimination and violence," which are inherently negative experiences. The use o...
Debug test result: {'sentiment_score': -0.8, 'sentiment_label': 'Very Negative', 'reasoning': 'The text describes a situation where the Catholic community experienced "discrimination and violence," which are inherently negative experiences. The use of these terms indicates a strong negative sentiment towards the events described, leading to a very negative overall sentiment.', 'api_success': True}


In [ ]:
# Debugged version of the sentiment analysis function
def add_sentiment_analysis_to_results_fixed(results_dict, analyze_original=True, analyze_cleaned=True, delay=1.0):
    """
     Add sentiment analysis to existing results using OpenAI
    
    Parameters:
    - results_dict: Dictionary of DataFrames from analyze_multiple_terms
    - analyze_original: bool, whether to analyze original content
    - analyze_cleaned: bool, whether to analyze cleaned content  
    - delay: float, delay between API calls to avoid rate limits
    
    Returns:
    - Updated results_dict with sentiment columns added
    Fixed version with better error handling
    """
    
    enhanced_results = {}
    
    for term, df in results_dict.items():
        print(f"\n{'='*60}")
        print(f"Adding sentiment analysis for term: '{term.upper()}'")
        print(f"{'='*60}")
        
        df_copy = df.copy()
        
        # Initialize sentiment columns
        if analyze_original:
            df_copy['original_sentiment_score'] = 0.0
            df_copy['original_sentiment_label'] = 'Pending'
            df_copy['original_sentiment_reasoning'] = 'Not analyzed yet'
            
        if analyze_cleaned:
            df_copy['cleaned_sentiment_score'] = 0.0
            df_copy['cleaned_sentiment_label'] = 'Pending'
            df_copy['cleaned_sentiment_reasoning'] = 'Not analyzed yet'
        
        # Analyze each document
        for idx, row in df_copy.iterrows():
            print(f"Analyzing document {idx+1}/{len(df_copy)} (Index: {row['document_index']})...")
            
            # Analyze original content
            if analyze_original:
                print("  - Analyzing original content...")
                original_sentiment = analyze_sentiment_with_openai_debug(  # Use debug version
                    row['full_original_document'], 
                    term
                )
                df_copy.at[idx, 'original_sentiment_score'] = original_sentiment['sentiment_score']
                df_copy.at[idx, 'original_sentiment_label'] = original_sentiment['sentiment_label']
                df_copy.at[idx, 'original_sentiment_reasoning'] = original_sentiment['reasoning']
                
                time.sleep(delay)
            
            # Analyze cleaned content
            if analyze_cleaned:
                print("  - Analyzing cleaned content...")
                cleaned_sentiment = analyze_sentiment_with_openai_debug(  # Use debug version
                    row['full_cleaned_document'], 
                    term
                )
                df_copy.at[idx, 'cleaned_sentiment_score'] = cleaned_sentiment['sentiment_score']
                df_copy.at[idx, 'cleaned_sentiment_label'] = cleaned_sentiment['sentiment_label']
                df_copy.at[idx, 'cleaned_sentiment_reasoning'] = cleaned_sentiment['reasoning']
                
                time.sleep(delay)
        
        enhanced_results[term] = df_copy
        print(f"Completed sentiment analysis for '{term}'")
    
    return enhanced_results

# Try the fixed version with just one term first
print("Testing fixed sentiment analysis with one term...")
test_single_term = {'catholic': all_results_option1['catholic'].head(2)}  # Just 2 documents
test_enhanced = add_sentiment_analysis_to_results_fixed(
    test_single_term, 
    analyze_original=True, 
    analyze_cleaned=False,  # Skip cleaned for now
    delay=2.0  # Longer delay
)


Testing fixed sentiment analysis with one term...

Adding sentiment analysis for term: 'CATHOLIC'
Analyzing document 1/2 (Index: 404)...
  - Analyzing original content...
Raw OpenAI response: Score: -0.9  
Label: Very Negative  
Reason: The text describes a highly negative perception of the Catholic faith, portraying it as satanic and associating it with disloyalty and potential treachery....
Analyzing document 2/2 (Index: 1110)...
  - Analyzing original content...
Raw OpenAI response: Score: -1.0  
Label: Very Negative  
Reason: The text describes a period of intense violence and fear experienced by the Catholic community in Lisburn during August 1920. It highlights the targeting a...
Completed sentiment analysis for 'catholic'


In [ ]:
# Full sentiment analysis with the fixed version
print("Running FULL sentiment analysis for Option 1...")
enhanced_option1_fixed = add_sentiment_analysis_to_results_fixed(
    all_results_option1, 
    analyze_original=True, 
    analyze_cleaned=True, 
    delay=2.0  # 2 second delay to be safe with rate limits
)

print("\n" + "="*80)
print("Option 1 analysis complete! Proceeding to Option 2...")
print("="*80)

enhanced_option2_fixed = add_sentiment_analysis_to_results_fixed(
    all_results_option2, 
    analyze_original=True, 
    analyze_cleaned=True, 
    delay=2.0  
)


Running FULL sentiment analysis for Option 1...

Adding sentiment analysis for term: 'CATHOLIC'
Analyzing document 1/5 (Index: 404)...
  - Analyzing original content...
Raw OpenAI response: Score: -0.9  
Label: Very Negative  
Reason: The text describes a highly negative perception of the Catholic faith, portraying it as satanic and associating it with disloyalty and potential treachery....
  - Analyzing cleaned content...
Raw OpenAI response: Score: -0.8  
Label: Very Negative  
Reason: The text contains predominantly negative language and sentiments towards the Catholic faith, describing it as "satanic" and "potentially untrustworthy." It...
Analyzing document 2/5 (Index: 1110)...
  - Analyzing original content...
Raw OpenAI response: Score: -1.0  
Label: Very Negative  
Reason: The text describes a period of intense violence and fear experienced by the Catholic community in Lisburn during August 1920. It highlights the targeting a...
  - Analyzing cleaned content...
Raw OpenAI respo

In [31]:
# Save enhanced results with sentiment analysis to CSV
def save_enhanced_results_to_csv(results_dict, output_dir, option_name):
    """
    Save the enhanced results with sentiment analysis to CSV files
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    for term, df in results_dict.items():
        filename = f"{output_dir}{option_name}_sentiment_analysis_{term.replace(' ', '_')}.csv"
        df.to_csv(filename, index=False)
        print(f"Saved enhanced results for '{term}' to {filename}")

# Save the enhanced results 
print("Saving enhanced Option 1 results with sentiment analysis...")
save_enhanced_results_to_csv(
    enhanced_option1_fixed, 
    '../outputs/analysis_results/no_interview/sentiments/option1/', 
    'option1'
)

print("Saving enhanced Option 2 results with sentiment analysis...")
save_enhanced_results_to_csv(
    enhanced_option2_fixed, 
    '../outputs/analysis_results/no_interview/sentiments/option2/', 
    'option2'
)


Saving enhanced Option 1 results with sentiment analysis...
Saved enhanced results for 'catholic' to ../outputs/analysis_results/no_interview/sentiments/option1/option1_sentiment_analysis_catholic.csv
Saved enhanced results for 'de valera' to ../outputs/analysis_results/no_interview/sentiments/option1/option1_sentiment_analysis_de_valera.csv
Saved enhanced results for 'éamon de valera' to ../outputs/analysis_results/no_interview/sentiments/option1/option1_sentiment_analysis_éamon_de_valera.csv
Saved enhanced results for 'sinn féin' to ../outputs/analysis_results/no_interview/sentiments/option1/option1_sentiment_analysis_sinn_féin.csv
Saving enhanced Option 2 results with sentiment analysis...
Saved enhanced results for 'violence' to ../outputs/analysis_results/no_interview/sentiments/option2/option2_sentiment_analysis_violence.csv
Saved enhanced results for 'support' to ../outputs/analysis_results/no_interview/sentiments/option2/option2_sentiment_analysis_support.csv
Saved enhanced res